# Quantara NanoGPT - Production Training (8-Layer, 512-dim)

GPU training notebook for the full production emotion model.  
Survives Colab disconnects via Google Drive checkpointing.

**Model:** 8 transformer layers, 512 embedding dim, 8 attention heads  
**Dataset:** 32-emotion taxonomy (9 families)  
**Integration:** Neural Workflow AI Engine, ML Training & Prediction Systems

In [ ]:
# ============================================================================
# Cell 1: Setup - Mount Drive, clone repo, install deps, detect GPU
# ============================================================================
import os
import subprocess

# Mount Google Drive for checkpoint persistence
from google.colab import drive
drive.mount('/content/drive')

DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/quantara-nanogpt-checkpoints'
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
print(f'Drive checkpoint dir: {DRIVE_CHECKPOINT_DIR}')

# Clone repo
REPO_DIR = '/content/quantara-nanoGPT'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/QEbellavita/quantara-nanoGPT.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)

# Install dependencies
!pip install -q torch numpy tiktoken

# Detect GPU type
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    print(f'GPU detected: {gpu_name} ({gpu_mem:.1f} GB)')
    if 'A100' in gpu_name:
        GPU_TIER = 'A100'
    elif 'T4' in gpu_name:
        GPU_TIER = 'T4'
    elif 'V100' in gpu_name:
        GPU_TIER = 'V100'
    else:
        GPU_TIER = 'OTHER'
else:
    gpu_name = 'CPU'
    GPU_TIER = 'CPU'
    print('WARNING: No GPU detected. Training will be very slow.')

print(f'GPU tier: {GPU_TIER}')
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

In [ ]:
# ============================================================================
# Cell 2: Data Preparation - Prepare train/val binary files
# ============================================================================
import os
os.chdir(REPO_DIR)

data_dir = os.path.join(REPO_DIR, 'data', 'quantara_emotion')
train_bin = os.path.join(data_dir, 'train.bin')
val_bin = os.path.join(data_dir, 'val.bin')

# Run data preparation if bins don't exist
if not os.path.exists(train_bin) or not os.path.exists(val_bin):
    print('Preparing training data...')
    !python data/quantara_emotion/prepare.py
else:
    print('Training data already prepared.')

# Verify data files
assert os.path.exists(train_bin), f'train.bin not found at {train_bin}'
assert os.path.exists(val_bin), f'val.bin not found at {val_bin}'

train_size = os.path.getsize(train_bin)
val_size = os.path.getsize(val_bin)
print(f'train.bin: {train_size:,} bytes ({train_size / 1024 / 1024:.2f} MB)')
print(f'val.bin:   {val_size:,} bytes ({val_size / 1024 / 1024:.2f} MB)')
print(f'Total tokens (approx): {(train_size + val_size) // 2:,}')

In [ ]:
# ============================================================================
# Cell 3: Config - Auto-detect GPU tier, set batch size and compile flag
# ============================================================================
import torch

# Auto-detect GPU tier and set optimal parameters
if GPU_TIER == 'A100':
    BATCH_SIZE = 64
    COMPILE = True
    DTYPE = 'bfloat16'
    GRAD_ACCUM = 2
    print('A100 detected: max performance settings')
elif GPU_TIER == 'T4':
    BATCH_SIZE = 32
    COMPILE = True
    DTYPE = 'float16'
    GRAD_ACCUM = 4
    print('T4 detected: balanced performance settings')
elif GPU_TIER == 'V100':
    BATCH_SIZE = 48
    COMPILE = True
    DTYPE = 'float16'
    GRAD_ACCUM = 3
    print('V100 detected: high performance settings')
else:
    BATCH_SIZE = 16
    COMPILE = False
    DTYPE = 'float32'
    GRAD_ACCUM = 8
    print('Default/CPU: conservative settings')

# Check if torch.compile is available (PyTorch 2.0+)
if COMPILE and not hasattr(torch, 'compile'):
    print('WARNING: torch.compile not available, disabling')
    COMPILE = False

print(f'\nTraining config:')
print(f'  batch_size:     {BATCH_SIZE}')
print(f'  compile:        {COMPILE}')
print(f'  dtype:          {DTYPE}')
print(f'  grad_accum:     {GRAD_ACCUM}')
print(f'  block_size:     512')
print(f'  n_layer:        8')
print(f'  n_embd:         512')
print(f'  n_head:         8')
tokens_per_iter = GRAD_ACCUM * BATCH_SIZE * 512
print(f'  tokens/iter:    {tokens_per_iter:,}')

In [ ]:
# ============================================================================
# Cell 4: Train - Resume from Drive checkpoint if available, run training
# ============================================================================
import os
import shutil
os.chdir(REPO_DIR)

OUT_DIR = os.path.join(REPO_DIR, 'out-quantara-emotion')
os.makedirs(OUT_DIR, exist_ok=True)

# Resume from Drive checkpoint if available
drive_ckpt = os.path.join(DRIVE_CHECKPOINT_DIR, 'ckpt.pt')
local_ckpt = os.path.join(OUT_DIR, 'ckpt.pt')

init_from = 'scratch'
if os.path.exists(drive_ckpt):
    print(f'Found Drive checkpoint: {drive_ckpt}')
    print(f'  Size: {os.path.getsize(drive_ckpt) / 1024 / 1024:.1f} MB')
    shutil.copy2(drive_ckpt, local_ckpt)
    init_from = 'resume'
    print('Resuming from Drive checkpoint.')
else:
    print('No Drive checkpoint found. Training from scratch.')

# Build training command
train_cmd = (
    f'python train.py config/train_quantara_emotion.py '
    f'--batch_size={BATCH_SIZE} '
    f'--compile={COMPILE} '
    f'--dtype={DTYPE} '
    f'--gradient_accumulation_steps={GRAD_ACCUM} '
    f'--init_from={init_from} '
    f'--device=cuda'
)

print(f'\nRunning: {train_cmd}\n')
!{train_cmd}

# Copy checkpoint to Drive after training completes
if os.path.exists(local_ckpt):
    print('\nSaving checkpoint to Google Drive...')
    shutil.copy2(local_ckpt, drive_ckpt)
    print(f'Checkpoint saved to {drive_ckpt}')
    print(f'  Size: {os.path.getsize(drive_ckpt) / 1024 / 1024:.1f} MB')
else:
    print('WARNING: No checkpoint found after training.')

In [ ]:
# ============================================================================
# Cell 5: Evaluate - Generate sample text for emotion-related prompts
# ============================================================================
import os
import pickle
import torch
from contextlib import nullcontext
os.chdir(REPO_DIR)

from model import GPTConfig, GPT

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16'
ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]
ctx = nullcontext() if device == 'cpu' else torch.amp.autocast(device_type='cuda', dtype=ptdtype)

# Load checkpoint
ckpt_path = os.path.join(REPO_DIR, 'out-quantara-emotion', 'ckpt.pt')
checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
model = GPT(gptconf)

state_dict = checkpoint['model']
unwanted_prefix = '_orig_mod.'
for k, v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
model.load_state_dict(state_dict)
model.eval()
model.to(device)

print(f'Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters')
print(f'Config: n_layer={gptconf.n_layer}, n_embd={gptconf.n_embd}, n_head={gptconf.n_head}')
print(f'Best val loss: {checkpoint.get("best_val_loss", "N/A")}')
print(f'Training iter: {checkpoint.get("iter_num", "N/A")}')

# Load encoder/decoder from meta.pkl
meta_path = os.path.join(REPO_DIR, 'data', 'quantara_emotion', 'meta.pkl')
if os.path.exists(meta_path):
    with open(meta_path, 'rb') as f:
        meta = pickle.load(f)
    stoi, itos = meta['stoi'], meta['itos']
    encode = lambda s: [stoi.get(c, 0) for c in s]
    decode = lambda l: ''.join([itos.get(i, '') for i in l])
    print(f'Vocab size: {meta["vocab_size"]}')
else:
    import tiktoken
    enc = tiktoken.get_encoding('gpt2')
    encode = lambda s: enc.encode(s, allowed_special={'<|endoftext|>'})
    decode = lambda l: enc.decode(l)
    print('Using GPT-2 tokenizer')

# Generate samples for emotion-related prompts
prompts = [
    'I feel a deep sense of joy when',
    'The anxiety was overwhelming because',
    'She felt calm and at peace after',
]

print('\n' + '='*60)
print('EMOTION PROMPT GENERATION')
print('='*60)

with torch.no_grad():
    with ctx:
        for prompt in prompts:
            print(f'\nPrompt: "{prompt}"')
            print('-' * 40)
            ids = encode(prompt)
            x = torch.tensor(ids, dtype=torch.long, device=device)[None, ...]
            y = model.generate(x, max_new_tokens=200, temperature=0.8, top_k=100)
            output = decode(y[0].tolist())
            print(output)
            print()

In [ ]:
# ============================================================================
# Cell 6: Export - Save production checkpoint to Drive
# ============================================================================
import os
import shutil
import torch
os.chdir(REPO_DIR)

local_ckpt = os.path.join(REPO_DIR, 'out-quantara-emotion', 'ckpt.pt')
production_ckpt = os.path.join(DRIVE_CHECKPOINT_DIR, 'ckpt_production.pt')

assert os.path.exists(local_ckpt), 'No trained checkpoint found. Run training first.'

# Copy as production checkpoint
shutil.copy2(local_ckpt, production_ckpt)

size_mb = os.path.getsize(production_ckpt) / (1024 * 1024)
print(f'Production checkpoint saved to: {production_ckpt}')
print(f'Size: {size_mb:.1f} MB')

# Verify the checkpoint loads correctly
ckpt = torch.load(production_ckpt, map_location='cpu')
print(f'\nCheckpoint contents:')
print(f'  model_args:     {ckpt["model_args"]}')
print(f'  iter_num:       {ckpt.get("iter_num", "N/A")}')
print(f'  best_val_loss:  {ckpt.get("best_val_loss", "N/A")}')
print(f'  state_dict keys: {len(ckpt["model"])}')

In [ ]:
# ============================================================================
# Cell 7: Retrain FusionHead - Train AttentionFusionHead on emotion labels
# ============================================================================
# Self-contained: loads production GPT embeddings (NOT sentence-transformers),
# trains AttentionFusionHead with CrossEntropyLoss, saves combined checkpoint.
#
# Connected to: Neural Workflow AI Engine, ML Training & Prediction Systems
# ============================================================================
import os
import glob
import pickle
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
os.chdir(REPO_DIR)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ---- Load production GPT model for embeddings ----
from model import GPTConfig, GPT

production_ckpt_path = os.path.join(DRIVE_CHECKPOINT_DIR, 'ckpt_production.pt')
assert os.path.exists(production_ckpt_path), 'Run Cell 6 first to export production checkpoint'

ckpt = torch.load(production_ckpt_path, map_location=device)
gptconf = GPTConfig(**ckpt['model_args'])
gpt_model = GPT(gptconf)
state_dict = ckpt['model']
unwanted_prefix = '_orig_mod.'
for k in list(state_dict.keys()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt_model.load_state_dict(state_dict)
gpt_model.eval()
gpt_model.to(device)

TEXT_DIM = gptconf.n_embd  # 512 for production model
POSE_DIM = 16
print(f'GPT model loaded: text_dim={TEXT_DIM}')

# ---- Load encoder from meta.pkl ----
meta_path = os.path.join(REPO_DIR, 'data', 'quantara_emotion', 'meta.pkl')
if os.path.exists(meta_path):
    with open(meta_path, 'rb') as f:
        meta = pickle.load(f)
    stoi = meta['stoi']
    encode = lambda s: [stoi.get(c, 0) for c in s]
else:
    import tiktoken
    enc = tiktoken.get_encoding('gpt2')
    encode = lambda s: enc.encode(s)

# ---- 32-emotion taxonomy ----
EMOTIONS = [
    'joy', 'excitement', 'enthusiasm', 'fun', 'gratitude', 'pride',
    'sadness', 'grief', 'boredom', 'nostalgia',
    'anger', 'frustration', 'hate', 'contempt', 'disgust', 'jealousy',
    'fear', 'anxiety', 'worry', 'overwhelmed', 'stressed',
    'love', 'compassion',
    'calm', 'relief', 'mindfulness', 'resilience', 'hope',
    'guilt', 'shame',
    'surprise',
    'neutral',
]
EMOTION_TO_IDX = {e: i for i, e in enumerate(EMOTIONS)}
NUM_EMOTIONS = len(EMOTIONS)
print(f'Emotions: {NUM_EMOTIONS}')

# ---- Load emotion-labeled CSV data ----
import csv

data_dir = os.path.join(REPO_DIR, 'data', 'quantara_emotion')
csv_files = glob.glob(os.path.join(data_dir, '*.csv'))

texts = []
labels = []

for csv_file in csv_files:
    try:
        with open(csv_file, 'r', encoding='utf-8', errors='ignore') as f:
            reader = csv.DictReader(f)
            headers = [h.lower().strip() for h in (reader.fieldnames or [])]
            # Find text and emotion columns
            text_col = None
            emotion_col = None
            for h in reader.fieldnames or []:
                hl = h.lower().strip()
                if hl in ('text', 'content', 'sentence', 'comment_text'):
                    text_col = h
                if hl in ('emotion', 'label', 'sentiment', 'emotions'):
                    emotion_col = h
            if text_col and emotion_col:
                count = 0
                for row in reader:
                    text = (row.get(text_col) or '').strip()
                    emotion = (row.get(emotion_col) or '').strip().lower()
                    if text and emotion in EMOTION_TO_IDX:
                        texts.append(text)
                        labels.append(EMOTION_TO_IDX[emotion])
                        count += 1
                if count > 0:
                    print(f'  Loaded {count:,} samples from {os.path.basename(csv_file)}')
    except Exception as e:
        print(f'  Skipping {os.path.basename(csv_file)}: {e}')

print(f'\nTotal labeled samples: {len(texts):,}')
assert len(texts) > 0, 'No emotion-labeled data found in CSV files'

# ---- Extract GPT embeddings ----
def get_gpt_embedding(text, max_len=128):
    """Get mean-pooled embedding from the GPT transformer."""
    ids = encode(text[:500])  # Truncate long texts
    ids = ids[:max_len]
    if len(ids) == 0:
        return torch.zeros(TEXT_DIM, device=device)
    x = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)
    with torch.no_grad():
        # Get transformer hidden states (before final lm_head)
        tok_emb = gpt_model.transformer.wte(x)
        pos_emb = gpt_model.transformer.wpe(torch.arange(x.size(1), device=device))
        h = gpt_model.transformer.drop(tok_emb + pos_emb)
        for block in gpt_model.transformer.h:
            h = block(h)
        h = gpt_model.transformer.ln_f(h)
        # Mean pool over sequence
        embedding = h.mean(dim=1).squeeze(0)
    return embedding

print('Extracting GPT embeddings...')
embeddings = []
batch_report = max(1, len(texts) // 10)
for i, text in enumerate(texts):
    emb = get_gpt_embedding(text)
    embeddings.append(emb)
    if (i + 1) % batch_report == 0:
        print(f'  {i + 1}/{len(texts)} ({100*(i+1)/len(texts):.0f}%)')

X_text = torch.stack(embeddings)  # (N, TEXT_DIM)
Y = torch.tensor(labels, dtype=torch.long, device=device)  # (N,)
print(f'Embeddings shape: {X_text.shape}')

# ---- Define AttentionFusionHead ----
class AttentionFusionHead(nn.Module):
    """Cross-modal attention fusion for emotion classification."""

    def __init__(self, text_dim=512, pose_dim=16, hidden_dim=128,
                 num_emotions=32, dropout=0.3, num_heads=4):
        super().__init__()
        self.text_proj = nn.Linear(text_dim, hidden_dim)
        self.pose_proj = nn.Linear(pose_dim, hidden_dim)
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=hidden_dim, num_heads=num_heads,
            dropout=dropout, batch_first=True
        )
        self.norm = nn.LayerNorm(hidden_dim)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_emotions),
        )
        # Text-only fallback path
        self.text_only_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_emotions),
        )

    def forward(self, text_emb, pose_emb=None):
        # text_emb: (B, text_dim), pose_emb: (B, pose_dim) or None
        t = self.text_proj(text_emb).unsqueeze(1)  # (B, 1, hidden)
        if pose_emb is not None:
            p = self.pose_proj(pose_emb).unsqueeze(1)  # (B, 1, hidden)
            kv = torch.cat([t, p], dim=1)  # (B, 2, hidden)
            attn_out, _ = self.cross_attn(t, kv, kv)
            out = self.norm(attn_out.squeeze(1) + t.squeeze(1))
            return self.classifier(out)
        else:
            return self.text_only_head(t.squeeze(1))

# ---- Train ----
EPOCHS = 20
LR = 1e-3
FUSION_BATCH_SIZE = 32

fusion_head = AttentionFusionHead(
    text_dim=TEXT_DIM, pose_dim=POSE_DIM,
    hidden_dim=128, num_emotions=NUM_EMOTIONS
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(fusion_head.parameters(), lr=LR)

# Simple train/val split
n = len(texts)
perm = torch.randperm(n)
split = int(0.9 * n)
train_idx, val_idx = perm[:split], perm[split:]

print(f'\nTraining AttentionFusionHead: {sum(p.numel() for p in fusion_head.parameters()):,} params')
print(f'Train: {len(train_idx):,}, Val: {len(val_idx):,}\n')

best_val_acc = 0.0
for epoch in range(EPOCHS):
    fusion_head.train()
    total_loss = 0.0
    correct = 0
    total = 0

    # Shuffle training indices
    shuffled = train_idx[torch.randperm(len(train_idx))]

    for i in range(0, len(shuffled), FUSION_BATCH_SIZE):
        batch_idx = shuffled[i:i+FUSION_BATCH_SIZE]
        x_batch = X_text[batch_idx]
        y_batch = Y[batch_idx]

        logits = fusion_head(x_batch)  # text-only path
        loss = criterion(logits, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(batch_idx)
        correct += (logits.argmax(dim=1) == y_batch).sum().item()
        total += len(batch_idx)

    train_acc = correct / total
    avg_loss = total_loss / total

    # Validation
    fusion_head.eval()
    with torch.no_grad():
        val_logits = fusion_head(X_text[val_idx])
        val_loss = criterion(val_logits, Y[val_idx]).item()
        val_acc = (val_logits.argmax(dim=1) == Y[val_idx]).float().mean().item()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.clone() for k, v in fusion_head.state_dict().items()}

    print(f'Epoch {epoch+1:2d}/{EPOCHS}  '
          f'loss={avg_loss:.4f}  train_acc={train_acc:.3f}  '
          f'val_loss={val_loss:.4f}  val_acc={val_acc:.3f}')

# Load best weights
fusion_head.load_state_dict(best_state)
print(f'\nBest val accuracy: {best_val_acc:.3f}')

# ---- Save combined checkpoint ----
combined_ckpt = {
    'gpt_model_args': ckpt['model_args'],
    'gpt_state_dict': ckpt['model'],
    'fusion_head_state_dict': fusion_head.state_dict(),
    'emotions': EMOTIONS,
    'meta': {
        'version': 2,
        'text_dim': TEXT_DIM,
        'pose_dim': POSE_DIM,
    },
    'best_val_acc': best_val_acc,
}

combined_path = os.path.join(DRIVE_CHECKPOINT_DIR, 'ckpt_production_with_fusion.pt')
torch.save(combined_ckpt, combined_path)
size_mb = os.path.getsize(combined_path) / (1024 * 1024)
print(f'\nCombined checkpoint saved to: {combined_path}')
print(f'Size: {size_mb:.1f} MB')
print(f'Meta: {combined_ckpt["meta"]}')